In [2]:
import pandas as pd
import os
import re

# 1. Path Configuration
excel_path = r"F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED.xlsx"

if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Could not find the Excel file at: {excel_path}")

# Load the file
df = pd.read_excel(excel_path)

# Clean up column names to prevent trailing/leading space KeyErrors
df.columns = df.columns.str.strip()

# Target column name variable (adjust if it's lowercase 'd' in your file)
target_col = 'Proposal Details' 

if target_col not in df.columns:
    raise KeyError(f"Could not find '{target_col}' column. Available columns are: {list(df.columns)}")

# 2. Define the Parsing Function
def extract_parivesh_details(text):
    # Fallback for empty/NaN cells
    if pd.isna(text):
        return pd.Series([None] * 5)
    
    text_str = str(text)
    
    # Using regex lookarounds to capture data between the known field labels
    clearance = re.search(r'Clearance Type:\s*(.*?)(?=\s*S/W No\.:|$)', text_str, re.IGNORECASE)
    sw_no     = re.search(r'S/W No\.\s*:\s*(.*?)(?=\s*Category:|$)', text_str, re.IGNORECASE)
    category  = re.search(r'Category\s*:\s*(.*?)(?=\s*Sector:|$)', text_str, re.IGNORECASE)
    sector    = re.search(r'Sector\s*:\s*(.*?)(?=\s*Date of Submission:|$)', text_str, re.IGNORECASE)
    date_sub  = re.search(r'Date of Submission\s*:\s*(.*?)$', text_str, re.IGNORECASE)
    
    # Extract match if found, strip trailing spaces, otherwise return None
    return pd.Series([
        clearance.group(1).strip() if clearance else None,
        sw_no.group(1).strip() if sw_no else None,
        category.group(1).strip() if category else None,
        sector.group(1).strip() if sector else None,
        date_sub.group(1).strip() if date_sub else None
    ])

# 3. Apply parsing to generate 5 new columns
print("Extracting data points from Proposal Details...")

new_cols = ['Clearance Type', 'S/W No.', 'Category', 'Sector', 'Date of Submission']
df[new_cols] = df[target_col].apply(extract_parivesh_details)

# 4. Save back to Excel
df.to_excel(excel_path, index=False)
print(f"Success! Extracted fields saved into columns: {new_cols}")

Extracting data points from Proposal Details...
Success! Extracted fields saved into columns: ['Clearance Type', 'S/W No.', 'Category', 'Sector', 'Date of Submission']


In [36]:
# Print all field names as a list
print(" DataFrame Fields:")
print(*df.columns.tolist(), sep="\n")

 DataFrame Fields:
S. No.
Proposal No.
Proposal Details
Project Name
Location
Project Proponent
Issuing Authority
Proposal Status
State_Value
Clearance Type
S/W No.
Category
Sector
Date of Submission


In [37]:
# 1. Path Configuration
excel_path = r"F:\Chimney Work\Marketing\Parivesh Work\backup\MOEFCC.xlsx"

if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Could not find the Excel file at: {excel_path}")

# Load the file
df2 = pd.read_excel(excel_path)

# Clean up column names to prevent trailing/leading space KeyErrors
df2.columns = df2.columns.str.strip()
# Print all field names as a list
print(" DataFrame Fields:")
print(*df2.columns.tolist(), sep="\n")

 DataFrame Fields:
S.No.
Proposal No
Category
Clearance Type
CAF Number
S/W Number
Project Name
Location
Project Proponent
Date of Submission
Proposal Status
Issuing Authority


In [39]:
# 1. Rename columns in df
df = df.rename(columns={
    'Proposal No.': 'Proposal No',
    'S. No.': 'SNo',
    'S/W No.': 'SW No'
})

# 2. Rename columns in df2
df2 = df2.rename(columns={
    'S.No.': 'SNo',
    'S/W Number': 'SW No'
})

print("Column names unified across both DataFrames!")

# Convert columns to sets for easy comparison
cols_df1 = set(df.columns)
cols_df2 = set(df2.columns)

# 1. Common fields in both
common_fields = cols_df1.intersection(cols_df2)

print("--- COMMON FIELDS IN BOTH DATAFRAMES ---")
if common_fields:
    print(*sorted(common_fields), sep="\n")
else:
    print("No matching fields found!")

print("\n--- UNIQUE TO FIRST FILE (df) ---")
print(*sorted(cols_df1 - cols_df2), sep="\n")

print("\n--- UNIQUE TO BACKUP FILE (df2) ---")
print(*sorted(cols_df2 - cols_df1), sep="\n")

Column names unified across both DataFrames!
--- COMMON FIELDS IN BOTH DATAFRAMES ---
Category
Clearance Type
Date of Submission
Issuing Authority
Location
Project Name
Project Proponent
Proposal No
Proposal Status
SNo
SW No

--- UNIQUE TO FIRST FILE (df) ---
Proposal Details
Sector
State_Value

--- UNIQUE TO BACKUP FILE (df2) ---
CAF Number


In [17]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)


SYSTEM_PROMPT = "create a new column 'Clearance Type' in the dataset using the 'Proposal Details' field. Extract details for 'Clearance Type' from the details given in 'Proposal details'. E.g., if 'Proposal Details' has value as "Clearance Type: Application for ToR (Category A, B1, and B2 Violation)/EC (Category B2) - Form 1 S/W No.: SW/181645/2024 Category: B2 Sector: Industrial Projects - 2 Date of Submission: 01/08/2024", the value for 'Clearance Type' should be "Application for ToR (Category A, B1, and B2 Violation)/EC (Category B2) - Form 1" - the extracted value starts from "Application for" and ends with Form no."

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": df}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)



SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (76502785.py, line 10)

In [ ]:
Step 1: Split the SEIAA file using LLM into 
    Clearance Type, 
    S/W No., 
    Category, 
    Sector, 
    Date of Submission

Prompt create a new column 'Clearance Type' in the attached file using the 'Proposal Details' field. Pull the details for 'Clearance Type' from the details given in 'Proposal details'. E.g., if the details given in the 'Proposal Details' are as "Clearance Type: Application for ToR (Category A, B1, and B2 Violation)/EC (Category B2) - Form 1 S/W No.: SW/181645/2024 Category: B2 Sector: Industrial Projects - 2 Date of Submission: 01/08/2024", the value for "Clearance Type" should be "Application for ToR (Category A, B1, and B2 Violation)/EC (Category B2) - Form 1". The value starts from "Application for" and ends with Form # 




In [16]:
import pandas as pd
from bs4 import BeautifulSoup
import re
import os

# -------------------------------------------------------------
# 1. Import the Excel File
# -------------------------------------------------------------
excel_path = r"F:\Chimney Work\Marketing\Parivesh Work\New\Combined File_FULL.xlsx"

if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Could not find the Excel file at: {excel_path}")

df = pd.read_excel(excel_path)

In [ ]:
#df.head() 

In [9]:
#df['Proposal Status'].unique()

pattern = 'granted|approved|accepted|Referred to EAC'
filtered_df = df[df['Proposal Status'].str.contains(pattern, case=False, na=False)]
filtered_out_df = df.drop(filtered_df.index)
filtered_df.shape[0]
filtered_out_df.shape[0]

34989

In [10]:
filtered_out_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 34989 entries, 0 to 88230
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Proposal No.           34989 non-null  object        
 1   Date of Submission     34989 non-null  datetime64[ns]
 2   Location               34989 non-null  object        
 3   Project Name           34989 non-null  object        
 4   Project Proponent      34988 non-null  object        
 5   Proposal Status        34989 non-null  object        
 6   Category               34927 non-null  object        
 7   Clearance Type         34989 non-null  object        
 8   Issuing Authority      34989 non-null  object        
 9   Flag (status)          28110 non-null  object        
 10  Flag (Clearance Type)  34989 non-null  object        
 11  Unnamed: 11            2 non-null      object        
 12  Unnamed: 12            0 non-null      object        
dtypes: dat

In [ ]:
#filtered_out_df['Proposal Status'].unique()

In [12]:
filtered_df['Clearance Type'].unique()

array(['Application for ToR (Category A, B1, and B2 Violation)/EC (Category B2) - Form 1',
       'Application for ToR (',
       'Application for amendment in ToR (for categories A & B1)/Amendment in EC (for category B2)- Form-3',
       'Application for Validity Extension of EC- Form-6',
       'Application for EC (',
       'Application for EC for Mining of Minor Minerals of Mine Lease (0-5 HA) - Form-2',
       'Application for Corrigendum Form-13',
       'Application for EC (Category A, B1, and B2 Violation)- Form 1',
       'Application for Amendment in EC- Form-4',
       'Application for amendment in ToR (for categories A & B1)/Amendment in EC (for',
       'Application for Transfer of EC- Form-7',
       'Application for Surrender of Environmental Clearance - Form -11',
       'Application for Splitting of Environmental Clearance - Form 12',
       'Application for Transfer of ToR - Form-8',
       'Application for Registration of Successful Bidder of ML for Transfer of EC- F

In [7]:
from google import genai

# The client automatically picks up GEMINI_API_KEY from your environment
client = genai.Client()

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Explain quantum computing to a five-year-old in two sentences."
)

print(response.text)

ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}